# IoT Sensor Data Trend Prediction — Aquaponics Fish Pond Water Quality

End-to-end pipeline: data ingestion -> cleaning -> feature engineering -> modeling -> evaluation/plots.
Runs as-is in Google Colab (pandas/numpy/scikit-learn/matplotlib are pre-installed).

**Dataset:** [Sensor Based Aquaponics Fish Pond Datasets](https://www.kaggle.com/datasets/ogbuokiriblessing/sensor-based-aquaponics-fish-pond-datasets) (Kaggle) — 12 pond files (`IoT_pond1.csv`..`IoT_pond12.csv`) from an ESP-32-controlled aquaponics fish pond: temperature, turbidity, dissolved oxygen, pH, ammonia, and nitrate sensors, plus fortnightly manual fish length/weight measurements.

**Target variable:** `dissolved_oxygen_mgl`, forecast 6 steps ahead (~30 minutes at the typical ~5-minute logging cadence). Dissolved Oxygen is the most safety-critical water-quality parameter for fish survival — published guidance flags risk below ~3 mg/L — so a 30-minute-ahead forecast is directly actionable (time to switch on an aerator).

If you don't upload a real pond CSV, Step 1 below auto-generates a synthetic dataset with the same six sensors and the same categories of real-world defects (WiFi dropouts, missing timestamps, hardware noise, out-of-bounds spikes), so every later cell still runs correctly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json as _json

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# ---------------- paths ----------------
RAW_DIR = Path("data/raw"); PROCESSED_DIR = Path("data/processed")
OUTPUTS_DIR = Path("outputs"); PLOTS_DIR = OUTPUTS_DIR / "plots"; MODELS_DIR = OUTPUTS_DIR / "models"
for d in (RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR, PLOTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
RAW_FILE = RAW_DIR / "IoT_pond1.csv"   # swap for any IoT_pond1.csv .. IoT_pond12.csv

# ---------------- canonical schema ----------------
COL_TIME = "created_at"
COL_TEMP = "temperature_c"
COL_TURBIDITY = "turbidity_ntu"
COL_DO = "dissolved_oxygen_mgl"
COL_PH = "ph"
COL_AMMONIA = "ammonia_ppm"
COL_NITRATE = "nitrate_ppm"
COL_LENGTH = "fish_length_cm"
COL_WEIGHT = "fish_weight_g"
SENSOR_COLS = [COL_TEMP, COL_TURBIDITY, COL_DO, COL_PH, COL_AMMONIA, COL_NITRATE]

COLUMN_KEYWORDS = {
    COL_TIME: ["created_at", "date", "time"], COL_TEMP: ["temp"], COL_TURBIDITY: ["turb"],
    COL_DO: ["oxygen", "dissolved"], COL_PH: ["ph"], COL_AMMONIA: ["ammonia"],
    COL_NITRATE: ["nitrate"], COL_LENGTH: ["length"], COL_WEIGHT: ["weight"],
}
PHYSICAL_BOUNDS = {COL_TEMP: (0, 45), COL_PH: (0, 14), COL_DO: (0, 20),
                    COL_TURBIDITY: (0, None), COL_AMMONIA: (0, None), COL_NITRATE: (0, None)}
DO_MORTALITY_RISK_MGL = 3.0
HORIZON_STEPS = 6

def detect_column(raw_columns, keywords):
    for col in raw_columns:
        if any(k in col.lower() for k in keywords):
            return col
    return None

def map_to_canonical_schema(df):
    '''Rename whatever the raw pond file calls its columns onto the canonical
    schema by keyword match -- robust to header spelling differing slightly
    across the 12 pond files.'''
    rename_map, remaining = {}, list(df.columns)
    for canonical, keywords in COLUMN_KEYWORDS.items():
        match = detect_column(remaining, keywords)
        if match:
            rename_map[match] = canonical
            remaining.remove(match)
    df = df.rename(columns=rename_map)
    missing = [c for c in [COL_TIME] + SENSOR_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Could not auto-detect required columns: {missing}. Raw columns: {list(df.columns)}")
    return df

def infer_sampling_freq(timestamps):
    '''Infer the dominant logging interval from real timestamp gaps, instead
    of trusting the nominal spec-sheet value -- real WiFi hardware drifts.'''
    diffs = timestamps.sort_values().diff().dropna()
    diffs = diffs[diffs > pd.Timedelta(0)]
    mode_freq = diffs.dt.round("s").mode()
    return mode_freq.iloc[0] if len(mode_freq) else pd.Timedelta(minutes=5)

def load_raw(path):
    df = pd.read_csv(path)
    df = map_to_canonical_schema(df)
    df[COL_TIME] = pd.to_datetime(df[COL_TIME], errors="coerce")
    df = df.dropna(subset=[COL_TIME]).sort_values(COL_TIME).set_index(COL_TIME)
    return df[~df.index.duplicated(keep="first")]

print("Config ready. RAW_FILE expected at:", RAW_FILE.resolve())

## Phase 1 — Data Ingestion

Loads a real pond export if present. Otherwise generates a synthetic-but-realistic
dataset with the same six sensors and the same defect categories (WiFi dropouts,
missing timestamps, out-of-bounds spikes, impossible negative readings) plus sparse
fortnightly fish length/weight, so the rest of the notebook is fully runnable.

In [ ]:
def generate_synthetic_pond(n_days=60, freq_minutes=5, seed=7):
    rng = np.random.default_rng(seed)
    n = n_days * 24 * 60 // freq_minutes
    base = pd.date_range("2021-06-01", periods=n, freq=f"{freq_minutes}min")
    timestamps = base + pd.to_timedelta(rng.normal(0, 20, n), unit="s")  # real hardware never logs on a clean grid
    t = np.arange(n); steps_per_day = 24 * 60 / freq_minutes

    temperature = 28 + 2.0 * np.sin(2 * np.pi * (t / steps_per_day - 0.3)) + rng.normal(0, 0.3, n)
    do_drift = np.cumsum(rng.normal(0, 0.01, n))
    do_drift -= pd.Series(do_drift).rolling(int(steps_per_day * 3), min_periods=1).mean().values
    dissolved_oxygen = np.clip(6.5 - 1.3 * np.sin(2 * np.pi * (t / steps_per_day - 0.3)) + do_drift + rng.normal(0, 0.15, n), 0.5, 12)
    ph = 7.2 + 0.3 * np.sin(2 * np.pi * t / (steps_per_day * 10)) + rng.normal(0, 0.05, n)

    ammonia = np.full(n, 0.15) + rng.normal(0, 0.03, n)
    feeding_events = rng.choice(n, size=n_days * 2, replace=False)
    for idx in feeding_events:
        decay_len = min(24, n - idx)
        ammonia[idx: idx + decay_len] += rng.uniform(0.3, 0.9) * np.exp(-np.arange(decay_len) / 8)

    nitrate = np.zeros(n); level = 5.0
    for i in range(n):
        level += rng.normal(0.01, 0.01)
        if rng.random() < 1 / (steps_per_day * 14):
            level = max(2.0, level * 0.3)
        nitrate[i] = level

    turbidity = np.abs(rng.normal(8, 2, n))
    turbidity[feeding_events] += rng.uniform(5, 25, len(feeding_events))

    df = pd.DataFrame({COL_TIME: timestamps, COL_TEMP: temperature, COL_TURBIDITY: turbidity,
                        COL_DO: dissolved_oxygen, COL_PH: ph, COL_AMMONIA: np.clip(ammonia, 0, None),
                        COL_NITRATE: nitrate})

    df[COL_LENGTH] = np.nan; df[COL_WEIGHT] = np.nan
    measurement_every = int(steps_per_day * 14)
    for i in range(0, n, measurement_every):
        growth_weeks = i / steps_per_day / 7
        df.loc[df.index[i], COL_LENGTH] = 8.0 + 0.4 * growth_weeks + rng.normal(0, 0.2)
        df.loc[df.index[i], COL_WEIGHT] = 25.0 + 8 * growth_weeks + rng.normal(0, 2)

    for _ in range(n_days // 4):  # WiFi dropout blocks
        start = rng.integers(0, n - 30); length = rng.integers(3, 30)
        col = rng.choice([COL_TEMP, COL_DO, COL_PH, COL_AMMONIA])
        df.loc[df.index[start:start + length], col] = np.nan
    drop_idx = rng.choice(df.index, size=int(0.03 * n), replace=False)  # missing timestamps
    df = df.drop(index=drop_idx)
    n_spikes = n_days // 3
    spike_idx = rng.choice(df.index, size=n_spikes, replace=False)
    df.loc[spike_idx, COL_DO] = df.loc[spike_idx, COL_DO] + rng.normal(0, 6, n_spikes)
    df.loc[spike_idx[: n_spikes // 2], COL_PH] = rng.uniform(0, 14, n_spikes // 2)
    neg_idx = rng.choice(df.index, size=n_days // 5, replace=False)
    df.loc[neg_idx, COL_TURBIDITY] = -rng.uniform(1, 20, len(neg_idx))
    return df

if RAW_FILE.exists():
    print(f"Found real dataset at {RAW_FILE} -- using it.")
else:
    print("No real CSV found -- generating synthetic aquaponics data for this run.")
    generate_synthetic_pond().to_csv(RAW_FILE, index=False)

df_raw = load_raw(RAW_FILE)
print(f"raw rows loaded: {len(df_raw):,}")
df_raw.head()

## Phase 2 — Data Cleaning

1. **Per-reading physical-plausibility rules** (domain knowledge): pH outside 0-14, DO above 20 mg/L, or negative turbidity/ammonia/nitrate are impossible for the hardware.
2. **Rolling median + MAD outlier detection**, deliberately permissive (high threshold) — a real ammonia spike right after feeding is a genuine signal, not noise, and shouldn't be erased by an overly strict filter.
3. **Resample (bin) onto a regular grid inferred from the data — NOT exact-match reindex.** Real WiFi-connected hardware never logs on a clean grid; reindexing onto an exact-timestamp grid would silently drop almost every jittered reading. Binning groups whatever falls in each interval and correctly leaves truly-empty intervals as NaN (missing timestamps).
4. **Gap-aware interpolation**: short gaps interpolated, long gaps dropped rather than fabricated.
5. **Fish length/weight forward-filled separately** — they're hand-measured every ~2 weeks, so most "missing" rows there are by design, not a sensor fault.

In [ ]:
MAX_SHORT_GAP_STEPS = 6
ROLLING_WINDOW_STEPS = 6
MAD_THRESHOLD = 8.0  # permissive on purpose -- keeps real feeding-time ammonia bumps mostly intact

def apply_physical_bounds(df):
    df = df.copy()
    n_before = df[SENSOR_COLS].notna().sum().sum()
    for col, (lo, hi) in PHYSICAL_BOUNDS.items():
        if lo is not None: df.loc[df[col] < lo, col] = np.nan
        if hi is not None: df.loc[df[col] > hi, col] = np.nan
    print(f"physical-bounds filter removed {n_before - df[SENSOR_COLS].notna().sum().sum()} implausible readings")
    return df

def remove_statistical_spikes(df, cols=SENSOR_COLS):
    df = df.copy(); total_flagged = 0
    for col in cols:
        med = df[col].rolling(ROLLING_WINDOW_STEPS, center=True, min_periods=3).median()
        abs_dev = (df[col] - med).abs()
        mad = abs_dev.rolling(ROLLING_WINDOW_STEPS, center=True, min_periods=3).median().replace(0, np.nan)
        is_spike = (0.6745 * abs_dev / mad) > MAD_THRESHOLD
        total_flagged += int(is_spike.sum())
        df.loc[is_spike, col] = np.nan
    print(f"rolling MAD filter flagged {total_flagged} statistical spikes as sensor noise")
    return df

def resample_to_regular_grid(df, freq):
    resampled = df.resample(freq).mean(numeric_only=True)
    n_missing = resampled[df.columns[0]].isna().sum()
    print(f"inferred sampling interval: {freq}")
    print(f"missing timestamp buckets detected: {n_missing} ({n_missing / max(len(resampled), 1):.2%} of buckets)")
    return resampled

def gap_aware_interpolate(df, cols=SENSOR_COLS):
    df = df.copy(); df["is_imputed"] = False
    for col in cols:
        was_na = df[col].isna()
        interpolated = df[col].interpolate(method="time", limit=MAX_SHORT_GAP_STEPS, limit_direction="both")
        df["is_imputed"] = df["is_imputed"] | (was_na & interpolated.notna())
        df[col] = interpolated
    return df

freq = infer_sampling_freq(pd.Series(df_raw.index))
df_clean = apply_physical_bounds(df_raw)
df_clean = remove_statistical_spikes(df_clean)
df_clean = resample_to_regular_grid(df_clean, freq)
df_clean = gap_aware_interpolate(df_clean)
for col in [COL_LENGTH, COL_WEIGHT]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].ffill()

n_before_drop = len(df_clean)
df_clean = df_clean.dropna(subset=SENSOR_COLS)
print(f"dropped {n_before_drop - len(df_clean)} rows with gaps too long to safely interpolate")
print(f"final cleaned rows: {len(df_clean):,}  |  {df_clean['is_imputed'].mean():.1%} short-gap interpolated")
df_clean.to_csv(PROCESSED_DIR / "02_cleaned.csv")
df_clean.head()

## Phase 3 — Feature Engineering

- **Lag features** (t-1, t-2, t-3, t-6, t-12) for all six sensors: water chemistry is autocorrelated minute-to-minute.
- **Rolling mean/std (1h, 3h)** for all six sensors: trend and volatility, not just instantaneous level.
- **Ammonia x Temperature interaction**: the toxic un-ionized ammonia fraction rises with both pH and temperature, so this product is a more biologically meaningful stress signal than either sensor alone.
- **Cyclical hour-of-day (sin/cos)**: DO and temperature both follow a diurnal cycle.
- **Target = Dissolved Oxygen shifted `-HORIZON_STEPS`** — shifting the target backward (not features forward) keeps every feature using only information available at prediction time.

In [ ]:
LAGS = [1, 2, 3, 6, 12]
ROLLING_WINDOWS = [6, 18]

def build_features(df):
    df = df.copy()
    for col in SENSOR_COLS:
        for lag in LAGS:
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)
    for col in SENSOR_COLS:
        for w in ROLLING_WINDOWS:
            df[f"{col}_roll_mean_{w}"] = df[col].shift(1).rolling(w).mean()
            df[f"{col}_roll_std_{w}"] = df[col].shift(1).rolling(w).std()
    df["ammonia_temp_interaction"] = df[COL_AMMONIA] * df[COL_TEMP]
    hour = df.index.hour + df.index.minute / 60.0
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["target_do"] = df[COL_DO].shift(-HORIZON_STEPS)
    n_before = len(df)
    df = df.dropna(subset=[c for c in df.columns if c not in (COL_LENGTH, COL_WEIGHT)])
    print(f"feature engineering: {n_before:,} -> {len(df):,} rows after dropping edge-window NaNs")
    return df

df_features = build_features(df_clean)
df_features.to_csv(PROCESSED_DIR / "03_features.csv")
df_features.head()

## Phase 4 — Modeling

- **Naive persistence baseline** — predict "no change"; the floor any real model must beat.
- **Random Forest Regressor** — primary trend-regression model: mixed-unit features without scaling, robust to noise, interpretable importances.
- **MLPRegressor** — small feed-forward neural net, the assessment's "sequential neural network" option in a TensorFlow/PyTorch-free environment. Swap for a Keras/PyTorch LSTM on the same windowed features if available locally.
- **Fish length/weight excluded from features** — they update every ~2 weeks, far slower than the 30-minute horizon, so they'd risk being a spurious "time period" fingerprint rather than real signal.

**Overfitting guards:** chronological train(70%)/val(15%)/test(15%) split (never shuffled), capacity caps on the Random Forest, early stopping on the neural net using a held-out *training*-period slice, train/val/test metrics reported side by side, plus a DO-risk-event recall check since DO is safety-critical here.

In [ ]:
FEATURE_COLS_EXCLUDE = {"target_do", "is_imputed", COL_LENGTH, COL_WEIGHT}
feature_cols = [c for c in df_features.columns if c not in FEATURE_COLS_EXCLUDE]

n = len(df_features)
train = df_features.iloc[: int(n * 0.70)]
val = df_features.iloc[int(n * 0.70): int(n * 0.85)]
test = df_features.iloc[int(n * 0.85):]
print(f"split sizes -> train: {len(train):,}  val: {len(val):,}  test: {len(test):,}")

X_train, y_train = train[feature_cols], train["target_do"]
X_val, y_val = val[feature_cols], val["target_do"]
X_test, y_test = test[feature_cols], test["target_do"]

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def evaluate(a, b): return {"rmse": rmse(a, b), "mae": float(mean_absolute_error(a, b))}

metrics = {}
metrics["baseline_persistence"] = {"test": evaluate(y_test, X_test["dissolved_oxygen_mgl_lag_1"])}

lin = LinearRegression().fit(X_train, y_train)
metrics["linear_regression"] = {"train": evaluate(y_train, lin.predict(X_train)),
                                 "val": evaluate(y_val, lin.predict(X_val)),
                                 "test": evaluate(y_test, lin.predict(X_test))}

rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5,
                            random_state=42, n_jobs=-1).fit(X_train, y_train)
metrics["random_forest"] = {"train": evaluate(y_train, rf.predict(X_train)),
                             "val": evaluate(y_val, rf.predict(X_val)),
                             "test": evaluate(y_test, rf.predict(X_test))}

scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)
mlp = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", early_stopping=True,
                    validation_fraction=0.15, n_iter_no_change=10, max_iter=500,
                    random_state=42).fit(X_train_s, y_train)
metrics["neural_net"] = {"train": evaluate(y_train, mlp.predict(X_train_s)),
                          "val": evaluate(y_val, mlp.predict(X_val_s)),
                          "test": evaluate(y_test, mlp.predict(X_test_s))}

test_predictions = pd.DataFrame({"actual": y_test, "pred_baseline": X_test["dissolved_oxygen_mgl_lag_1"],
                                  "pred_linear": lin.predict(X_test),
                                  "pred_random_forest": rf.predict(X_test),
                                  "pred_neural_net": mlp.predict(X_test_s)}, index=test.index)
test_predictions.to_csv(OUTPUTS_DIR / "test_predictions.csv")

actual_risk = y_test < DO_MORTALITY_RISK_MGL
predicted_risk = pd.Series(rf.predict(X_test), index=y_test.index) < DO_MORTALITY_RISK_MGL
if actual_risk.any():
    recall = (actual_risk & predicted_risk).sum() / actual_risk.sum()
    print(f"low-DO risk events in test set: {int(actual_risk.sum())}  |  Random Forest recall: {recall:.1%}")
    metrics["low_do_risk_recall"] = float(recall)
else:
    print("no low-DO risk events in this test window (synthetic run) -- meaningful once run on real pond data.")

feature_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
feature_importance.to_csv(OUTPUTS_DIR / "feature_importance.csv")
with open(OUTPUTS_DIR / "model_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)

print(_json.dumps(metrics, indent=2))
print("\ntrain->val RMSE gap (Random Forest): "
      f"{metrics['random_forest']['val']['rmse'] - metrics['random_forest']['train']['rmse']:.3f} mg/L "
      "(large gap would indicate overfitting)")

## Phase 5 — Evaluation & Visualization

In [ ]:
rows = [{"model": m, "split": s, **v} for m, splits in metrics.items() for s, v in
         (splits.items() if isinstance(splits, dict) else [])]
metrics_table = pd.DataFrame(rows)
metrics_table.to_csv(OUTPUTS_DIR / "metrics_table.csv", index=False)
metrics_table

In [ ]:
def plot_actual_vs_predicted(predictions, model_col, label):
    fig, ax = plt.subplots(figsize=(13, 5))
    window = predictions.iloc[-2000:]
    ax.plot(window.index, window["actual"], label="Actual Dissolved Oxygen (mg/L)", color="#1f77b4", linewidth=1.2)
    ax.plot(window.index, window[model_col], label=f"Predicted ({label})", color="#d62728", linewidth=1.0, alpha=0.85)
    ax.axhline(DO_MORTALITY_RISK_MGL, color="#ff7f0e", linestyle="--", linewidth=1.2,
               label=f"Fish-stress risk threshold ({DO_MORTALITY_RISK_MGL} mg/L)")
    ax.set_title(f"Historical vs. Predicted Dissolved Oxygen -- {label}")
    ax.set_xlabel("Time"); ax.set_ylabel("Dissolved Oxygen (mg/L)"); ax.legend()
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"actual_vs_predicted_{label.lower().replace(' ', '_')}.png", dpi=140)
    plt.show()

plot_actual_vs_predicted(test_predictions, "pred_random_forest", "Random Forest")

In [ ]:
def plot_residuals(predictions, model_col, label):
    residuals = predictions["actual"] - predictions[model_col]
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(predictions.index, residuals, color="#9467bd", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"Residuals over Test Period -- {label}"); ax.set_ylabel("Actual - Predicted (mg/L)")
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"residuals_{label.lower().replace(' ', '_')}.png", dpi=140)
    plt.show()

plot_residuals(test_predictions, "pred_random_forest", "Random Forest")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
feature_importance.head(12).sort_values().plot(kind="barh", ax=ax, color="#2ca02c")
ax.set_title("Random Forest -- Top Feature Importances (DO forecast)"); ax.set_xlabel("Importance")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "feature_importance.png", dpi=140)
plt.show()

## Notes for the demo video

- Walk through: dataset/target (top) -> Phase 1-3 printed reports (call out the resample-not-reindex fix and the ammonia x temperature feature) -> Phase 4 metrics (baseline vs Random Forest vs neural net, the train/val RMSE gap, and the low-DO risk recall) -> Phase 5 plots.
- All outputs are saved under `outputs/` (and `data/processed/`) so they can be committed to the GitHub repo alongside this notebook.
- Swap in a real `IoT_pondN.csv` (Phase 1) before recording the final run — everything downstream is schema-driven via keyword column detection and needs no changes.